# Modeling - Daniel Lopez

For this section, I am comparing several models to predict whether a client will default next month. I started with a simple baseline model, then compared it to logistic regression, decision tree, random forest, gradient boosting, and a neural network model.

In [12]:
# Basic packages
import pandas as pd
import numpy as np

# Modeling tools
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

# Model evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report)

# For retraining the selected model later
from sklearn.base import clone

RANDOM_SEED = 123

In [13]:
# Load the dataset from the GitHub repo
url = "https://raw.githubusercontent.com/cameronmoran23/ADS504-Group6-FinalProject/main/data.csv"

data_df = pd.read_csv(url)

print(data_df.shape)
data_df.head()

(30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [14]:
# Rename columns so they are easier to work with
rename_dict = {
    "ID": "id",
    "LIMIT_BAL": "limit_bal",
    "SEX": "sex",
    "EDUCATION": "education",
    "MARRIAGE": "marriage",
    "AGE": "age",
    "PAY_0": "pay_1",
    "PAY_2": "pay_2",
    "PAY_3": "pay_3",
    "PAY_4": "pay_4",
    "PAY_5": "pay_5",
    "PAY_6": "pay_6",
    "BILL_AMT1": "bill_amt1",
    "BILL_AMT2": "bill_amt2",
    "BILL_AMT3": "bill_amt3",
    "BILL_AMT4": "bill_amt4",
    "BILL_AMT5": "bill_amt5",
    "BILL_AMT6": "bill_amt6",
    "PAY_AMT1": "pay_amt1",
    "PAY_AMT2": "pay_amt2",
    "PAY_AMT3": "pay_amt3",
    "PAY_AMT4": "pay_amt4",
    "PAY_AMT5": "pay_amt5",
    "PAY_AMT6": "pay_amt6",
    "default payment next month": "default"}
data_df = data_df.rename(columns=rename_dict)

# Clean up a few unusual category values
# Education should mainly be 1, 2, 3, or 4, so I am grouping 0, 5, and 6 into "other"
data_df["education"] = data_df["education"].replace({0: 4, 5: 4, 6: 4})

# Marriage should mainly be 1, 2, or 3, so I am grouping 0 into "other"
data_df["marriage"] = data_df["marriage"].replace({0: 3})

# Make sure the target is an integer
data_df["default"] = data_df["default"].astype(int)
data_df.head()

,id,limit_bal,sex,education,marriage,age,pay_1,pay_2,pay_3,pay_4,...,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,default
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [15]:
# ID is just an identifier, so I am not using it as a predictor
X = data_df.drop(columns=["id", "default"])
y = data_df["default"]

# Check the target balance
# 0 = no default, 1 = default
print(y.value_counts())
print(y.value_counts(normalize=True))

default
0    23364
1     6636
Name: count, dtype: int64
default
0    0.7788
1    0.2212
Name: proportion, dtype: float64


In [16]:
# First split off the test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y)

# Then split the remaining data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25,
    random_state=RANDOM_SEED, stratify=y_train_val)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print("Test set:", X_test.shape)

Training set: (18000, 23)
Validation set: (6000, 23)
Test set: (6000, 23)


In [17]:
# These are categorical columns based on the dataset documentation
categorical_cols = ["sex", "education", "marriage"]

# Everything else is treated as numeric
numeric_cols = [col for col in X.columns if col not in categorical_cols]

# Numeric columns are filled if needed and scaled
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())])

# Categorical columns are filled if needed and one-hot encoded
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

# Combine the numeric and categorical preprocessing
preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)])

In [18]:
# I am comparing a simple baseline model against several stronger models
models = {
    "Dummy Baseline": DummyClassifier(strategy="most_frequent"),

    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_SEED),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=5,
        class_weight="balanced",
        random_state=RANDOM_SEED),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_SEED),

    "Neural Network / MLP": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        max_iter=100,
        early_stopping=True,
        random_state=RANDOM_SEED)}

In [19]:
# This function trains a model and returns the main metrics
def evaluate_model(model_name, model, X_train, y_train, X_eval, y_eval):
    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", model)])

    # Train the model
    pipe.fit(X_train, y_train)

    # Predict classes
    y_pred = pipe.predict(X_eval)

    # Predict probabilities for AUC metrics
    y_proba = pipe.predict_proba(X_eval)[:, 1]

    results = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_eval, y_pred),
        "Precision": precision_score(y_eval, y_pred, zero_division=0),
        "Recall": recall_score(y_eval, y_pred, zero_division=0),
        "F1 Score": f1_score(y_eval, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, y_proba),
        "PR-AUC": average_precision_score(y_eval, y_proba)}
    return results, pipe

# Train and evaluate each model
model_results = []
trained_models = {}

for name, model in models.items():
    results, fitted_pipe = evaluate_model(
        name, model, X_train,
        y_train, X_val, y_val)

    model_results.append(results)
    trained_models[name] = fitted_pipe

# Put results into a table
results_df = pd.DataFrame(model_results)

# Sort by F1 because default prediction needs a balance between precision and recall
results_df.sort_values(by="F1 Score", ascending=False)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC
2,Decision Tree,0.719500,0.415238,0.657121,0.508900,0.756658,0.509605
1,Logistic Regression,0.697667,0.388762,0.641296,0.484073,0.726977,0.498093
4,Gradient Boosting,0.820167,0.665333,0.376036,0.480501,0.786102,0.553005
5,Neural Network / MLP,0.820000,0.673212,0.361718,0.470588,0.775875,0.535790
3,Random Forest,0.814500,0.649860,0.349661,0.454679,0.772483,0.540262
0,Dummy Baseline,0.778833,0.000000,0.000000,0.000000,0.500000,0.221167


In [20]:
# Pick the best model based on validation F1 score
best_model_name = results_df.sort_values(by="F1 Score", ascending=False).iloc[0]["Model"]

print("Best model based on validation F1 score:", best_model_name)

results_df.sort_values(by="F1 Score", ascending=False)

Best model based on validation F1 score: Decision Tree


,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC
2,Decision Tree,0.719500,0.415238,0.657121,0.508900,0.756658,0.509605
1,Logistic Regression,0.697667,0.388762,0.641296,0.484073,0.726977,0.498093
4,Gradient Boosting,0.820167,0.665333,0.376036,0.480501,0.786102,0.553005
5,Neural Network / MLP,0.820000,0.673212,0.361718,0.470588,0.775875,0.535790
3,Random Forest,0.814500,0.649860,0.349661,0.454679,0.772483,0.540262
0,Dummy Baseline,0.778833,0.000000,0.000000,0.000000,0.500000,0.221167


In [21]:
# Retrain the selected model on the combined train + validation data
best_base_model = models[best_model_name]

final_model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", clone(best_base_model))])

final_model.fit(X_train_val, y_train_val)

# Final test predictions
y_test_pred = final_model.predict(X_test)
y_test_proba = final_model.predict_proba(X_test)[:, 1]

# Final metrics
final_results = {
    "Model": best_model_name,
    "Accuracy": accuracy_score(y_test, y_test_pred),
    "Precision": precision_score(y_test, y_test_pred, zero_division=0),
    "Recall": recall_score(y_test, y_test_pred, zero_division=0),
    "F1 Score": f1_score(y_test, y_test_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_test_proba),
    "PR-AUC": average_precision_score(y_test, y_test_proba)}

pd.DataFrame([final_results])

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC,PR-AUC
0,Decision Tree,0.780667,0.503724,0.560663,0.53067,0.751685,0.510512


In [22]:
# Confusion matrix shows the correct and incorrect predictions
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_test_pred))

Confusion Matrix:
[[3940  733]
 [ 583  744]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.84      0.86      4673
           1       0.50      0.56      0.53      1327

    accuracy                           0.78      6000
   macro avg       0.69      0.70      0.69      6000
weighted avg       0.79      0.78      0.78      6000

